# How AI Agents Think and Act

## Course Plan:

- Lecture 1 — From LLM to Agent: The Simplest Possible Loop — DONE
- Lecture 2 — Memory and RAG — DONE
- Lecture 3 — Graphs and Planning — DONE
- **Lecture 4 — Multi-Agent Systems** ← you are here

---

Prerequisites:
- `.env` file with `OPENAI_API_KEY` and `TAVILY_API_KEY`
- `pip install langgraph tavily-python openai python-dotenv`

# Lecture 4 — Multi-Agent Systems: Division of Labor

**Central question:** When does one agent become multiple agents, and why?

- Lectures 1–3 built progressively smarter *single* agents: a bare loop, a memory-augmented loop, and a graph-structured planner. Each improvement made the agent more capable — but all the intelligence still lived in one process, one prompt, one loop.
- Today we ask: *what happens when a task is too large, too varied, or too conflicted for one agent to handle well?*

The answer is **decomposition**: an **orchestrator** agent that plans and assigns work, and **worker** agents that execute specialized tasks. The interface between them — the thing that replaces function signatures and type systems — is a **natural language task description**.

> *"The interface between agents is a sentence."* — This is the payoff of the whole course.

---
# Setup

In [ ]:
# Standard imports and path setup
import os, sys, json
from typing import TypedDict, Literal, Annotated
import operator
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

# LangGraph core — Send is the primitive for dynamic fan-out
from langgraph.graph import StateGraph, END
from langgraph.types import Send
from IPython.display import Image, display

# API clients
from openai import OpenAI
from tavily import TavilyClient

openai_client: OpenAI = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
tavily_client: TavilyClient = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

print("Setup complete.")

---
# Part 1 — Why One Agent Is Not Always Enough

Before building a multi-agent system, we should understand what problem it solves. Let's look at three failure modes that appear when a single agent is asked to do too much.

In [ ]:
# Demonstrate role conflict: one prompt forced to be critic AND advocate simultaneously
conflicted_prompt: str = (
    "You are a helpful assistant. Write a persuasive argument that remote work "
    "increases productivity. Then critique that argument as a skeptic. "
    "Then recommend a policy based on both perspectives."
)

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": conflicted_prompt}],
    temperature=0.7,
)
output: str = response.choices[0].message.content or ""
print(output[:800])
print("\n... (output continues)")

<details>
<summary>Details: <strong>Three failure modes of single-agent systems</strong></summary>

**1. Role conflict.** One model cannot genuinely hold two opposing perspectives at once. Ask it to argue both sides and it hedges, averages, or drifts toward one position. A human team assigns "devil's advocate" to a different person for a reason.

**2. Context window exhaustion.** A single agent accumulates all context in one conversation: tools, results, reasoning, history. Complex tasks generate thousands of tokens of intermediate state. The model's attention degrades at long contexts; older information gets "forgotten" in practice even when it's technically present.

**3. No parallelism.** A single agent runs one step at a time. If three sub-tasks are independent, it still executes them serially. A multi-agent system can dispatch them simultaneously.

All three are arguments for **decomposition**: break the task, assign the pieces, recombine the results.
</details>

---
# Part 2 — The Orchestrator / Worker Pattern

The simplest multi-agent architecture has two roles:

- **Orchestrator** — receives the goal, decomposes it into tasks, and **spawns one worker per task**.
- **Worker** — receives exactly **one** task (a natural language description), executes it, and returns one result.

Each worker is an isolated agent: it sees only its own task, nothing else.

```
User Goal
    │
    ▼
[Orchestrator]
    │   │   │
    ▼   ▼   ▼        ← three workers launched in parallel
  [W1] [W2] [W3]
    │   │   │
    └───┴───┘
        │
    [Synthesizer]
        │
    Final Answer
```

LangGraph makes this fork literal: the orchestrator emits **`Send` objects** — one per task — and LangGraph launches each worker as a separate concurrent branch. The `results` field in state uses a **reducer** to merge their outputs as they arrive.

In [ ]:
# Two state schemas: one for the shared graph, one private to each worker

# Shared graph state — orchestrator writes tasks, workers write results, synthesizer reads both
class MultiAgentState(TypedDict):
    goal: str                                         # user's original request
    tasks: list[str]                                  # orchestrator-generated task list
    results: Annotated[list[str], operator.add]       # worker results, merged via reducer
    final_answer: str                                 # synthesized output

# Per-worker state — each worker instance sees only its own task and id
class WorkerState(TypedDict):
    task: str        # a single natural language task description
    worker_id: int   # for labelling output

print("MultiAgentState and WorkerState defined.")

<details>
<summary>Details: <strong>Two state schemas and why workers get their own</strong></summary>

The graph has two distinct state types:

**`MultiAgentState`** is the shared graph state — it lives for the entire run. The orchestrator writes to `tasks`; each worker writes one entry to `results`; the synthesizer reads both.

**`WorkerState`** is private to a single worker invocation. It carries only `task` and `worker_id` — nothing from the global state. This isolation is intentional: a worker that cannot read `goal` or other workers' results cannot accidentally depend on them.

The `Annotated[list[str], operator.add]` on `results` is what makes fan-out safe. Without the reducer, two workers returning `{"results": [...]}` simultaneously would overwrite each other. With `operator.add`, LangGraph concatenates each worker's list into the growing shared list, regardless of arrival order.
</details>

---
# Part 3 — The Orchestrator Node

The orchestrator receives the goal and produces a list of tasks — then immediately **dispatches one worker per task** using `Send`. It does not run the workers itself; it hands control to LangGraph, which launches them in parallel.

In [ ]:
# Orchestrator: decompose the goal into tasks, then fan-out via Send
def orchestrator_node(state: MultiAgentState) -> dict:
    goal: str = state["goal"]
    print(f"[orchestrator] decomposing goal: '{goal}'")

    # Ask the model to produce exactly 3 independent research tasks
    system_prompt: str = (
        "You are a research coordinator. Given a research goal, decompose it into "
        "exactly 3 specific, self-contained research tasks. "
        "Each task should be answerable by searching the web independently. "
        "Output ONLY a JSON array of 3 strings, like: "
        '[\"task one\", \"task two\", \"task three\"]. '
        "No other text."
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Research goal: {goal}"},
        ],
        temperature=0.2,
    )
    raw: str = (response.choices[0].message.content or "").strip()
    tasks: list[str] = json.loads(raw)

    print(f"[orchestrator] generated {len(tasks)} tasks:")
    for i, task in enumerate(tasks, 1):
        print(f"  {i}. {task}")

    # Store tasks in shared state (so synthesizer can inspect them later)
    return {"tasks": tasks}

# Fan-out routing function: called after orchestrator, returns one Send per task
# Each Send tells LangGraph: "invoke 'worker' with this private WorkerState"
def dispatch_workers(state: MultiAgentState) -> list[Send]:
    return [
        Send("worker", WorkerState(task=task, worker_id=i))
        for i, task in enumerate(state["tasks"], 1)
    ]

print("orchestrator_node and dispatch_workers defined.")

<details>
<summary>Details: <strong>dispatch_workers as a routing function — how it differs from Lecture 3</strong></summary>

In Lecture 3, `add_conditional_edges` took a routing function that returned a **single string** — the name of the one next node to activate. It was an exclusive choice: `"write"` or `"search"` or `"give_up"`. Exactly one branch fired.

`dispatch_workers` is also passed to `add_conditional_edges`, but it returns something different: a **list of `Send` objects**. LangGraph activates *all* of them, simultaneously.

Two things make `Send` richer than a node name string:

1. **It is not a choice — it is a group.** All items in the list fire. There is no "pick one"; the runtime forks into as many branches as the list contains. The number of branches is not fixed at graph-definition time — it is determined at runtime by whatever the function returns.

2. **Each item is a call request, not just a destination.** `Send("worker", WorkerState(...))` is closer to a delegate call in C# — it names the function to invoke *and* the argument to pass. A plain string edge says "go to this node and pass the current shared state." A `Send` says "invoke this node with *this specific private state*, independent of the shared state." Each worker branch gets its own isolated snapshot.

The shape this produces at runtime:

```
routing fn returns [Send("worker", s1), Send("worker", s2), Send("worker", s3)]
                          ↓                    ↓                    ↓
                    worker(s1)           worker(s2)           worker(s3)   ← all concurrent
                          ↓                    ↓                    ↓
                    {"results": [...]}   {"results": [...]}   {"results": [...]}
                                    ↘         ↓         ↙
                                      reducer merges all
                                            ↓
                                       synthesizer
```

The synthesizer starts only after every branch has written its result. LangGraph handles that barrier; you write none of that synchronization code.
</details>

<details>
<summary>Details: <strong>What is Send?</strong></summary>

`Send(node_name, state)` is LangGraph's primitive for **dynamic fan-out**. Instead of a fixed edge `orchestrator → worker`, you return a list of `Send` objects from a routing function — one per branch you want to create.

LangGraph launches all of them concurrently. Each branch runs the named node with its own private state snapshot. When all branches complete, their return values are merged back into the shared graph state using the field reducers.

This is what makes the worker pattern real: the orchestrator does not loop over tasks sequentially. It hands a list of `Send` objects to the runtime, and the runtime schedules all workers in parallel. The number of workers is determined at runtime by the orchestrator's output — not hardcoded in the graph definition.

Compare to Lecture 3: there, a conditional edge pointed to a single next node. Here, a routing function returns *multiple* `Send` objects — each going to the same `"worker"` node but with a different private state. The same node runs three times simultaneously with three different inputs.
</details>

---
# Part 4 — The Worker Node

Each worker instance receives a `WorkerState` with exactly **one task**. It knows nothing about the goal, the other tasks, or other workers. It searches, summarizes, and writes one result back into the shared `results` list.

In [ ]:
# Worker node: operates on WorkerState — one task, isolated from everything else
def worker_node(state: WorkerState) -> dict:
    task: str = state["task"]
    worker_id: int = state["worker_id"]
    print(f"[worker-{worker_id}] executing: '{task[:70]}...'")

    # Search for relevant information about this specific task
    search_response: dict = tavily_client.search(query=task, max_results=4)
    results: list[dict] = search_response.get("results", [])
    search_text: str = "\n".join(r.get("content", "") for r in results)

    if not search_text.strip():
        result: str = f"[worker-{worker_id}] No results found for: {task}"
        return {"results": [result]}

    # Summarize findings into a self-contained paragraph
    prompt: str = (
        f"You are a research assistant. Answer the following research task "
        f"concisely (2–3 sentences) based on the search results provided.\n\n"
        f"Task: {task}\n\n"
        f"Search results:\n{search_text[:2000]}\n\n"
        f"Answer:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    answer: str = (response.choices[0].message.content or "").strip()
    result = f"[Task: {task}]\n{answer}"
    print(f"[worker-{worker_id}] done. ({len(result)} chars)")

    # Return a single-item list — the reducer appends it to the shared results
    return {"results": [result]}

print("worker_node defined. Each instance handles exactly one task.")

<details>
<summary>Details: <strong>Worker isolation — enforced by the state type</strong></summary>

The worker function's parameter type is `WorkerState`, not `MultiAgentState`. It literally cannot read `goal`, `tasks`, or other workers' results — those fields don't exist in its state.

This is as close as a language-interface system gets to a type-enforced contract. The worker's isolation is structural, not just conventional.

**What the worker does not know:**
- The original user goal
- How many other workers are running
- What the other workers are doing or have found

**What this enables:**
- Each worker can be tested independently with a single `WorkerState` dict.
- A worker failure (exception, empty result) affects only its one entry in `results`.
- The orchestrator can dispatch 2 workers or 20 with no change to the worker code.

**The one trade-off:** workers cannot coordinate. If finding 1 is "the answer is X" and finding 2 is "the answer is not X", neither worker knows about the contradiction. That is the synthesizer's problem.
</details>

### Debugging a Worker independently

Because `worker_node` takes a plain `WorkerState` dict, we can call it directly — no graph, no orchestrator, no setup.

In [ ]:
# Call worker_node directly — same function the graph will invoke, no graph required
test_worker_state: WorkerState = {
    "task": "What is the vanishing gradient problem in deep neural networks?",
    "worker_id": 99,
}

test_result: dict = worker_node(test_worker_state)

print("Return value:", test_result)
print("\nResult string:")
print(test_result["results"][0])

---
# Part 5 — The Synthesizer Node

The synthesizer receives all worker results and produces a single coherent answer. This is the only node that sees both the original goal and everything the workers found.

In [ ]:
# Synthesizer node: combine all worker results into a final coherent answer
def synthesizer_node(state: MultiAgentState) -> dict:
    goal: str = state["goal"]
    results: list[str] = state["results"]
    print(f"[synthesizer] combining {len(results)} worker results")

    # Format all results as a numbered list for the model
    combined_results: str = "\n\n".join(
        f"Finding {i}:\n{r}" for i, r in enumerate(results, 1)
    )

    # Prompt sees the original goal and all worker outputs — nothing else
    prompt: str = (
        f"You are a senior research analyst. The following findings were gathered "
        f"by independent research workers to address a research goal. "
        f"Synthesize them into a single, well-structured answer (4–6 sentences).\n\n"
        f"Original goal: {goal}\n\n"
        f"Worker findings:\n{combined_results}\n\n"
        f"Synthesized answer:"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    final_answer: str = (response.choices[0].message.content or "").strip()
    print(f"[synthesizer] answer ready ({len(final_answer)} chars)")
    return {"final_answer": final_answer}

print("synthesizer_node defined.")

<details>
<summary>Details: <strong>Why bulleted list, not a Python list?</strong></summary>

Look at how `combined_results` is built:

```python
combined_results: str = "\n\n".join(
    f"Finding {i}:\n{r}" for i, r in enumerate(results, 1)
)
```

We have a perfectly good Python `list[str]` — `results` — and we immediately flatten it into a single numbered string before putting it in the prompt. Why not pass the list directly?

Because the synthesizer is not a Python function. It is a language model. **It does not receive data structures — it receives text.** A Python list would have to be serialized into some string representation anyway; doing it explicitly with `"Finding 1:\n..."` gives us full control over what the model reads.

This is the linguistic nature of the technology made concrete: even when the data *inside our Python program* is a typed, structured list, the moment it crosses the boundary into the model it must become **prose**. The model's "type system" is reading comprehension, not a schema validator.

The formatting choice — numbered headings, blank lines between findings — is not aesthetic. It is load-bearing. A wall of text with no visual separation degrades synthesis quality. The prompt is an interface, and whitespace is part of its grammar.

This applies everywhere in agent systems: any time data flows *into* a model, the developer is responsible for serializing it into a form the model can reason about well. There is no automatic **marshalling**. You are always writing a document, not calling a function.
</details>

<details>
<summary>Details: <strong>Why a separate synthesizer?</strong></summary>

The synthesizer could be eliminated — the orchestrator could collect results and write the final answer itself. But giving it a dedicated node has two advantages:

1. **Separation of concerns.** Decomposition (orchestrator) and integration (synthesizer) are cognitively distinct tasks. A model prompted for one does it better than a model prompted for both.

2. **Inspectability.** The synthesizer receives all worker results as an explicit, readable list in the state dict. At any point you can inspect `state["results"]` to see exactly what the synthesizer was working from — no hidden context.

The synthesizer is the only node in this graph that sees the full picture. Everything before it is specialized. This mirrors how research teams work: analysts gather, editors synthesize.
</details>

---
# Part 6 — Assemble the Multi-Agent Graph

The graph has three nodes: `orchestrator`, `worker`, `synthesizer`. The edge from `orchestrator` is a **conditional edge** that calls `dispatch_workers` — which returns a list of `Send` objects, one per task. LangGraph sees this and forks execution into parallel branches.

In [ ]:
# Assemble the multi-agent graph with parallel fan-out
ma_builder: StateGraph = StateGraph(MultiAgentState)

# Register nodes
ma_builder.add_node("orchestrator", orchestrator_node)
ma_builder.add_node("worker", worker_node)
ma_builder.add_node("synthesizer", synthesizer_node)

# Entry point
ma_builder.set_entry_point("orchestrator")

# After orchestrator: dispatch_workers returns a list of Send objects — one per task
# LangGraph launches all workers in parallel; results merge via the reducer
ma_builder.add_conditional_edges("orchestrator", dispatch_workers, ["worker"])

# After all workers complete: proceed to synthesizer
ma_builder.add_edge("worker", "synthesizer")
ma_builder.add_edge("synthesizer", END)

# Compile and render — the diagram shows the fan-out shape
multi_agent_graph = ma_builder.compile()
display(Image(multi_agent_graph.get_graph().draw_mermaid_png()))

<details>
<summary>Details: <strong>Reading the diagram — fan-out and fan-in</strong></summary>

The diagram shows the **fan-out / fan-in** shape: `orchestrator` branches into multiple `worker` nodes, which all converge at `synthesizer`.

This is structurally different from anything in Lecture 3. There, conditional edges pointed to a *single* next node. Here, a routing function returns *multiple* `Send` objects, and LangGraph renders this as multiple parallel edges from `orchestrator` to `worker`.

At runtime, the number of parallel branches equals the number of tasks the orchestrator produced — which is determined by the model at runtime, not by the graph definition at compile time. This is **dynamic parallelism**: the graph's shape changes with each invocation depending on the orchestrator's output.

The `synthesizer` node starts only after *all* worker branches have completed. LangGraph handles this barrier automatically — you do not need to write any synchronization code.
</details>

<details>
<summary>Details: <strong>Q1: Why doesn't the diagram show parallel branches?</strong></summary>

LangGraph's `draw_mermaid_png()` renders the *static graph definition* — the nodes and edge types registered at compile time. The `Send`-based fan-out is dynamic: the number of branches only exists at runtime, determined by the orchestrator's output. The static diagram therefore shows a **single `worker` box** with one edge from `orchestrator`, because that is all that is known at definition time. Mermaid has no representation for "this edge may fork into N copies at runtime."

To visualise the actual execution you would use `stream_mode="updates"` and print each node invocation as it fires — the output would show `[worker-1]`, `[worker-2]`, `[worker-3]` appearing in some interleaved order, making the parallelism visible in the log rather than the diagram. In a live class demo, streaming output is more legible than a static graph anyway: the audience watches the workers race each other in real time.
</details>

<details>
<summary>Details: <strong>Q2: Can the synthesizer re-synthesize incrementally as each worker finishes?</strong></summary>

**Yes** — this is called the **reflection** pattern.

Instead of a hard barrier ("wait for all, then synthesize once"), you wire the graph so each incoming worker result triggers a synthesizer node that reads whatever has accumulated so far, produces a draft answer, and optionally decides whether to request more work.

In LangGraph terms, the synthesizer node becomes part of a loop: each `Send` completion fans into it individually, and a routing function decides whether to keep updating the answer or declare it done. The result feels like a thinking agent that refines its view as new evidence arrives rather than batch-processing a completed pile.

The cost is complexity: the synthesizer must handle partial state gracefully, and the routing function must have a sensible termination condition.
</details>



<details>
<summary>Details: <strong>Q3: What if one worker throws an exception?</strong></summary>

By default, an unhandled exception in any node propagates up and aborts the entire graph invocation — the other workers' results are discarded and the synthesizer never runs. For production systems you have two options.

- The first is to wrap the worker body in a `try/except` and return a sentinel result string (e.g. `"[worker-2] failed: timeout"`) instead of raising — the synthesizer then receives N results, some of which are error messages, and can reason about them in natural language.

- The second is LangGraph's **built-in retry and fallback mechanism** at the graph level, which lets you re-invoke a failed node automatically before propagating the error. For a course demo the `try/except` approach is sufficient and keeps the failure visible in the state dict rather than hidden in framework machinery.
</details>


In [ ]:
# Run the full multi-agent system — workers execute in parallel
GOAL: str = "How does gradient descent work and why does it sometimes get stuck in local minima?"

initial_state: MultiAgentState = {
    "goal": GOAL,
    "tasks": [],
    "results": [],
    "final_answer": "",
}

print(f"Goal: {GOAL}\n")
final_state: MultiAgentState = multi_agent_graph.invoke(initial_state)

print("\n--- Final Answer ---")
print(final_state["final_answer"])

<details>
<summary>Details: <strong>How did the miracle happened?</strong></summary>

**How did three genuinely different answers emerge?**

Looking at the output, something feels surprising: three workers ran the same function against the same topic, yet each returned a meaningfully different perspective — one on mechanics, one on failure modes, one on remedies. Where did that variation come from? Not from random sampling, not from temperature noise, and not from any trick in the code.

It came from the **task strings**.

The orchestrator produced three distinct sentences:
- *"Explain the basic principles of gradient descent and its mathematical formulation."*
- *"Investigate the concept of local minima in optimization problems and how they affect gradient descent."*
- *"Explore techniques and strategies used to avoid local minima in gradient descent algorithms."*

Each sentence focused the worker's search query and shaped its summarization prompt in a completely different direction. The three results feel like three different experts answering three different questions — because they were. The orchestrator wrote the questions; the workers answered them.

This is the linguistic nature of the system made visible. There is no branching logic in the code that says "worker 1 covers theory, worker 2 covers failure modes." That structure exists entirely in natural language, inside the strings the orchestrator generated. The model — given only the goal and a prompt asking for independent subtasks — invented the conceptual decomposition on its own.

This is also why changing the goal produces a completely different structure, with no code change: the decomposition is not hardcoded anywhere. It is generated fresh each time, from language.
</details>

<details>
<summary>Details: <strong>Multiple Perspectives Merge</strong></summary>

The orchestrator's system prompt contains a fixed instruction: *"decompose the goal into exactly 3 specific, self-contained research tasks."* That is one perspective — a general decomposition strategy, baked in at design time, knowing nothing about gradient descent.

The user's goal is a completely separate perspective: *"How does gradient descent work and why does it sometimes get stuck in local minima?"* — a specific question from a specific domain, arriving at runtime, knowing nothing about the decomposition strategy.

**TASK:** Merge two independent perspectives and provide a relevant response:

Neither perspective contains the other. The system prompt says nothing about gradient descent. The user goal says nothing about how to decompose a research question. They are **orthogonal**: written independently, stored separately, with no shared vocabulary or structure.

Yet the model fused them into three subtasks that are simultaneously:
- consistent with the decomposition instruction (independent, web-searchable, self-contained), and
- semantically faithful to the user's question (covering mechanics, failure modes, and remedies — the natural structure of that specific topic).

Think about what it would take to achieve this fusion algorithmically. You would need to parse the intent of the user query, identify its domain, retrieve a matching decomposition schema, project the schema onto the domain's concept graph, and produce subtasks that satisfy constraints from both sides — all without any predefined schema for "gradient descent questions." The engineering effort would be enormous, and it would still break on any query that didn't fit the schema.

The language model did it in one forward pass, as a side effect of predicting the next token.

This is not a trick specific to this example. It is the general capability that makes language the right medium for agent interfaces: **natural language composes perspectives that were never designed to meet.** The system prompt and the user message are two independent texts. The model's job is to read both and produce output that honors both — and it does this by default, for any pair of texts, without any explicit merge logic written by the developer.

Every prompt in every agent system in this course relies on this capability. It is so routine that it becomes invisible. It is worth pausing to notice it.
</details>

**Pause here!!!**

Look at what actually flowed between the orchestrator and the workers. The tasks list contains plain English sentences. There is no schema, no type, no contract — just a string that a worker must interpret and execute.

This is the moment the course has been building toward.

In [ ]:
# Inspect the intermediate state — tasks and worker results
print("PERSPECTIVE #1: User task:")
print(f"  goal: {final_state["goal"]}")

print("\nPERSPECTIVE #2: Hard-coded word 'THREE'")
print(f"  Split the reasoning into THREE branches")

print("\nPERSPECTIVES #3: (a), (b), (c) tasks generated by LLM:")
for i, task in enumerate(final_state["tasks"], 1):
    print(f"  {i}. {task}")

print(f"\nPERSPECTIVES #4: Web results ({len(final_state['results'])} total):")
for i, result in enumerate(final_state["results"], 1):
    print(f"\n  --- Result {i} ---")
    print(f"  {result[:300]}")

print(f"\nPERSPECTIVE #5: Hard-coded word 'SYNTHESIZE' (could be 'object', 'explain', ...):")
print(f"  Synthesis: {final_state["final_answer"][:50]}...")

<details>
<summary>Details: <strong>Perspectives Merging Flow</strong></summary>

Every node in this graph is a perspective merge — a point where two or more independent sources of meaning collide inside a language model and produce something neither source could produce alone. There are three such collisions in a single run.

---

**Level 1 — Planning: structure meets topic**

```
Orchestrator system prompt          User goal
"decompose into THREE               "How does gradient descent work
 independent tasks"                  and why does it get stuck?"
        │                                      │
        └──────────────┬───────────────────────┘
                       ▼
              model merges them
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
  "explain          "local         "techniques
   mechanics"        minima"        to escape"
```

The only "parameter" the orchestrator contributes is the word **THREE** and the constraint **independent**. The topic structure — mechanics, failure modes, remedies — comes entirely from the user's sentence. Neither side knew about the other. The model invented a decomposition that satisfies both simultaneously.

---

**Level 2 — Search: task intent meets the open web**

Each worker performs its own merge independently:

```
Worker task string                  Web search results
"Investigate local minima           [arxiv paper on loss landscapes]
 and how they affect                [blog post on saddle points]
 gradient descent"                  [lecture notes on convexity]
        │                                      │
        └──────────────┬───────────────────────┘
                       ▼
              model merges them
                       │
                       ▼
         "Local minima are points where..."
           (2–3 sentence answer grounded
            in retrieved evidence)
```

The task string is a sentence from the planning level. The web results are raw, unstructured, written by strangers with no knowledge of this system. The model reads both and produces a focused, relevant paragraph — again, no explicit merge logic.

---

**Level 3 — Synthesis: operator meets findings**

```
Synthesizer system prompt           Three worker results
"Synthesize into a                  [finding 1: mechanics]
 single well-structured             [finding 2: local minima]
 answer (4–6 sentences)"            [finding 3: techniques]
        │                                      │
        └──────────────┬───────────────────────┘
                       ▼
              model merges them
                       │
                       ▼
         One coherent answer covering
         all three aspects in the
         requested style and length
```

The word **Synthesize** is a parameter. Replace it with *"Critique"* and the same three findings produce a skeptical assessment. Replace it with *"Explain to a 10-year-old"* and the same findings produce simplified prose. The synthesizer's prompt is a lens — it determines not what material exists but how it is processed. The model applies the lens to the material without any code change.

---

**The pattern across all three levels:**

Each merge has the same structure: a **static perspective** (baked into a prompt at design time) collides with a **dynamic input** (arriving at runtime) and produces output that is faithful to both. No merge function was written. No schema maps one to the other. The model's ability to read two independent texts and honor both simultaneously is the mechanism — and it works at every level of the stack.

This is what makes the architecture composable: you can change the operator at any level independently of the others. Change "THREE" to "FIVE" in the orchestrator prompt, and five workers fan out. Change "Synthesize" to "Debate" in the synthesizer prompt, and the final answer becomes adversarial. Neither change touches any other node. Each perspective is a dial; the language model is the mixer.
</details>

<details>
<summary>Details: <strong>Language as API — what this buys and what it costs</strong></summary>

**What it buys:**
- **Flexibility.** The orchestrator can decompose any goal into any task, without registering new function signatures. A new kind of task needs no code change — just a different sentence.
- **Generality.** Workers built for one domain can often handle adjacent domains, because natural language generalizes. A "research assistant" worker can follow instructions about machine learning, history, or cooking.
- **Evolvability.** You can change what the orchestrator requests by changing its prompt, not its code.

**What it costs:**
- **Brittleness.** A badly-phrased task produces a bad result with no error. The worker does not say "I don't understand" — it tries to comply and may produce confident nonsense.
- **No contract enforcement.** In classical software, if the API signature changes, the caller breaks at compile time. In a language-interface system, the caller (orchestrator) silently starts sending tasks the worker misunderstands.
- **Debugging is hard.** When a worker produces a wrong result, is the fault in the orchestrator's task description, the worker's prompt, the search results, or the synthesis? All of these are strings. None of them have a stack trace.
- **Security breach.** When the agent is given access to too powerful tools the ambiguity can cause serious damages.

This trade-off — flexibility at the cost of reliability — is the central tension in all current agent systems.
</details>

---
# Part 7 — Specialized Workers

So far all workers run the same function. The power of multi-agent systems grows when workers are **specialized** — each built with a different prompt, different tools, or a different model.

Let's build a system with two distinct worker types: a **researcher** (uses Tavily to find facts) and a **critic** (uses only the model to find weaknesses and counterarguments).

In [ ]:
# State for the specialized worker system
class SpecializedState(TypedDict):
    claim: str                                   # the claim to investigate
    research: str                                # factual findings from the researcher
    critique: str                                # weaknesses found by the critic
    verdict: str                                 # final balanced assessment

# Researcher worker: search for evidence supporting or explaining the claim
def researcher_node(state: SpecializedState) -> dict:
    claim: str = state["claim"]
    print(f"[researcher] investigating: '{claim[:60]}...'")

    # Search for factual information about the claim
    search_response: dict = tavily_client.search(query=claim, max_results=5)
    results: list[dict] = search_response.get("results", [])
    search_text: str = "\n".join(r.get("content", "") for r in results)

    # Summarize what the evidence actually says
    prompt: str = (
        f"You are a careful researcher. Based on the search results, summarize "
        f"what evidence exists about the following claim. Be factual and neutral.\n\n"
        f"Claim: {claim}\n\nSearch results:\n{search_text[:2000]}\n\nResearch summary:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    research: str = (response.choices[0].message.content or "").strip()
    print(f"[researcher] done ({len(research)} chars)")
    return {"research": research}

# Critic worker: find weaknesses, counterarguments, and caveats — no search
def critic_node(state: SpecializedState) -> dict:
    claim: str = state["claim"]
    research: str = state["research"]
    print(f"[critic] challenging the claim and research...")

    # The critic sees only the claim and the research — its job is to push back
    prompt: str = (
        f"You are a skeptical critic. Your job is to identify weaknesses, "
        f"counterarguments, and important caveats for the following claim and research. "
        f"Do not agree — find problems.\n\n"
        f"Claim: {claim}\n\nResearch summary:\n{research}\n\nCritique:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )
    critique: str = (response.choices[0].message.content or "").strip()
    print(f"[critic] done ({len(critique)} chars)")
    return {"critique": critique}

# Verdict node: balance research and critique into a final assessment
def verdict_node(state: SpecializedState) -> dict:
    claim: str = state["claim"]
    research: str = state["research"]
    critique: str = state["critique"]
    print(f"[verdict] synthesizing research and critique...")

    prompt: str = (
        f"You are a balanced analyst. Given the research findings and the critique, "
        f"write a fair final assessment of the claim. Acknowledge both supporting "
        f"evidence and valid objections.\n\n"
        f"Claim: {claim}\n\n"
        f"Research:\n{research}\n\n"
        f"Critique:\n{critique}\n\n"
        f"Final assessment:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    verdict: str = (response.choices[0].message.content or "").strip()
    print(f"[verdict] done ({len(verdict)} chars)")
    return {"verdict": verdict}

print("Specialized nodes defined: researcher, critic, verdict.")

<details>
<summary>Details: <strong>Specialization through prompting</strong></summary>

The researcher, critic, and verdict nodes all call the same `gpt-4o-mini` model. The specialization is entirely in the system prompt.

This is how most "specialized agents" work in practice — not different models, but different instructions to the same model. The role is defined by the prompt, not the architecture.

When does this break down? When the required specialization is deep domain knowledge, not just a different perspective. A "legal reasoning" agent and a "code generation" agent would genuinely benefit from fine-tuned or domain-specific models. Role-via-prompt works when the task is within the base model's capability; it fails when the task requires knowledge the base model doesn't have.

Also notice: the critic never searches. It can only challenge based on the research it receives. This is deliberate — we wanted a role that reasons, not retrieves. This is an architectural decision embedded in the node design.
</details>

In [ ]:
# Assemble the specialized-worker graph
spec_builder: StateGraph = StateGraph(SpecializedState)

# Register the three specialized nodes
spec_builder.add_node("researcher", researcher_node)
spec_builder.add_node("critic", critic_node)
spec_builder.add_node("verdict", verdict_node)

# researcher must run before critic (critic reads research)
spec_builder.set_entry_point("researcher")
spec_builder.add_edge("researcher", "critic")
spec_builder.add_edge("critic", "verdict")
spec_builder.add_edge("verdict", END)

specialized_graph = spec_builder.compile()
display(Image(specialized_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Run the specialized system on a contested claim
CLAIM: str = "Neural scaling laws mean that making models larger will continue to improve intelligence indefinitely."

spec_result: SpecializedState = specialized_graph.invoke({
    "claim": CLAIM,
    "research": "",
    "critique": "",
    "verdict": "",
})

print("\n--- Research ---")
print(spec_result["research"])
print("\n--- Critique ---")
print(spec_result["critique"])
print("\n--- Verdict ---")
print(spec_result["verdict"])

---
# Part 8 — Honest Limitations

We have built a working multi-agent system. Before closing, we should be honest about what it is and what it isn't. Agents are powerful but fragile. The field is moving fast and the abstractions are still settling.

In [ ]:
# Deliberately break the orchestrator to show what failure looks like
print("=== Demonstrating failure: ambiguous goal ===\n")

ambiguous_state: MultiAgentState = {
    "goal": "make it better",   # no context — what is 'it'?
    "tasks": [],
    "results": [],
    "final_answer": "",
}

try:
    broken_result: MultiAgentState = multi_agent_graph.invoke(ambiguous_state)
    print("Orchestrator produced tasks anyway:")
    for task in broken_result["tasks"]:
        print(f"  - {task}")
    print()
    print("Observation: the orchestrator hallucinated a context.")
    print("No error was raised. The system appeared to work.")
    print("This is a silent failure — the most dangerous kind.")
except Exception as e:
    print(f"Error: {e}")

<details>
<summary>Details: <strong>Five limitations to know before deploying agents</strong></summary>

**1. Silent failures.** An agent never refuses a badly-specified task. It hallucinate context, invents assumptions, and returns a confident-sounding answer. There is no runtime exception for "I didn't understand the goal."

**2. Error propagation.** In a pipeline, each node's output becomes the next node's input. A bad orchestrator decomposition produces bad worker tasks, which produce bad results, which produce a bad synthesis. The error compounds silently through every stage.

**3. No shared memory between agents.** Each node call is stateless except for what's explicitly in the state dict. If a worker discovers something important that wasn't anticipated as a state field, it cannot communicate it to other workers.

**4. Cost and latency multiply.** Every LLM call adds latency and cost. A 3-task system with search makes at minimum 5 model calls (1 orchestrator + 3 workers + 1 synthesizer) and 3 Tavily calls. Complex multi-agent systems can easily spend 10× what a single-agent solution would.

**5. Prompt brittleness compounds.** Each agent has its own prompt. A small wording change in the orchestrator prompt can change the decomposition, which changes what the workers receive, which changes the synthesis. The system has no integration tests for this.

These are not reasons to avoid agents — they are reasons to build them carefully, add evaluation, and maintain human oversight for high-stakes tasks.
</details>

---
# Recap

| Concept | Where it appeared |
|---|---|
| Why multiple agents | Part 1 — role conflict, context exhaustion, serial bottleneck |
| Orchestrator / worker pattern | Parts 2–6 — decompose → fan-out → synthesize |
| `Send` and dynamic fan-out | Part 3 — one `Send` per task, workers run in parallel |
| Two state schemas | Part 2 — `MultiAgentState` (shared) vs. `WorkerState` (isolated) |
| `Annotated` reducer | Part 2 — `operator.add` merges parallel worker results safely |
| Specialized workers | Part 7 — same model, different prompts, different roles |
| Silent failure | Part 8 — ambiguous goal produces confident hallucination |

**The course payoff:**

In Lecture 1 we looked at a JSON string carrying a tool-call and said: *"This is the interface between the model and the tool."*

Today we looked at a plain English sentence inside a `Send` object and said: *"This is the interface between the orchestrator and the worker."*

Four lectures later, the string is still the interface. In classical software, interfaces are enforced by type systems. In agent systems, the interface is natural language — and the contract is in the prompt.

---

*The field is evolving rapidly. The frameworks will change. The models will improve. But this fundamental tension — flexibility through language versus reliability through types — will be with us for a long time.*